## gics_mapping.csv 관련 Insight

### 🔍 GICS 계층 구조 검증 코드
목적

본 스크립트는 gics_mapping.csv 내 각 코드(sector, industry_group, industry, sub_industry)가
올바른 계층적 구조를 따르고 있는지 자동으로 검증하기 위해 작성되었습니다.

GICS(Global Industry Classification Standard)는 일반적으로 다음과 같은 4단계 구조를 가집니다:

sector → industry_group → industry → sub_industry
예시: 25 → 2510 → 251010 → 25101010


즉, 오른쪽으로 갈수록 왼쪽 코드에 2자리씩 숫자가 추가되는 규칙을 따라야 합니다.
본 코드는 이러한 규칙을 위배하는 데이터를 탐지하여,
데이터 전처리 단계에서 오류를 조기에 발견하고 품질을 보장하는 것이 목적입니다.

In [4]:
import pandas as pd

# 1️⃣ CSV 파일 불러오기
df = pd.read_csv("../output/gics_mapping.csv")

# 2️⃣ 각 컬럼을 문자열로 변환 (소수점, .0 제거)
for col in ["sector", "industry_group", "industry", "sub_industry"]:
    df[col] = df[col].astype(str).str.replace(".0", "", regex=False)

# 3️⃣ 규칙 위반 행 찾기
violations = []

for _, row in df.iterrows():
    s, ig, ind, si = row["sector"], row["industry_group"], row["industry"], row["sub_industry"]
    symbol = row["symbol"]

    # 규칙 검증
    if not ig.startswith(s):
        violations.append((symbol, "industry_group", ig, s))
    elif not ind.startswith(ig):
        violations.append((symbol, "industry", ind, ig))
    elif not si.startswith(ind):
        violations.append((symbol, "sub_industry", si, ind))

# 4️⃣ 결과 출력
if violations:
    print("⚠️ 규칙 위반 데이터가 발견되었습니다:\n")
    for v in violations:
        print(f"symbol={v[0]}, 오류컬럼={v[1]}, 값={v[2]}, 기준={v[3]}")
else:
    print("✅ 모든 데이터가 규칙을 잘 따르고 있습니다.")


✅ 모든 데이터가 규칙을 잘 따르고 있습니다.


## 10분 VWAP과 일별 VWAP의 관계 검증
### 목적

이 코드는 vwap.csv와 close_XXXX_XXXX.csv 파일 간의 관계를 분석하여
각 데이터가 어떤 의미의 가격 정보를 담고 있는지를 명확히 규명하기 위해 작성되었다.

vwap.csv: 하루 전체 거래의 거래량 가중평균가격(VWAP)

close_XXXX_XXXX.csv: 각 10분 구간별 거래량 가중평균가격(10분 VWAP)

즉, 10분 단위 데이터가 단순 체결가가 아닌
해당 구간의 VWAP임을 데이터적으로 검증하는 것이 목적이다.

### 검증 방법

- 일별 VWAP 불러오기

vwap.csv에서 특정 종목(symbol)과 날짜(date_key)에 해당하는 VWAP을 기준값으로 설정한다.

- 10분 VWAP 파일 로드

close_0900_0910.csv부터 close_1520_1530.csv까지 모든 파일을 읽고,
동일 종목과 날짜의 VWAP 값을 추출한다.

- 10분 VWAP 평균 계산

추출한 10분 VWAP 값을 단순 평균하여 하루 VWAP과 비교한다.

구간별 거래량이 없으므로 단순 평균을 근사치로 사용한다.

- 결측 구간 확인

각 구간 파일에 값이 존재하지 않는 경우를 확인하여 데이터 누락 여부를 점검한다.

### 결과 해석

10분 VWAP의 단순 평균값이 일별 VWAP(vwap.csv)과 약 1% 내외의 차이를 보였다.

일부 구간(예: 15:20–15:30)에 결측이 있었으나 전체적으로 일관된 패턴을 보였다.

이는 10분 단위 파일의 값이 각 구간의 VWAP이며,
vwap.csv가 하루 전체의 VWAP으로 계산되었음을 시사한다.

### 결론

close_XXXX_XXXX.csv 파일은 각 시간대의 VWAP(거래량 가중평균가격)을 의미한다.

vwap.csv는 이들 구간 VWAP의 거래량 가중 평균으로 계산된 일간 VWAP이다.

In [3]:
import pandas as pd
import glob
import os
import re
import numpy as np
import math

symbol = "AEGQRD"
date_key = "20200103"  # 인덱스에 있는 날짜 형식(예: 20200103)

def read_pivot_csv(path: str) -> pd.DataFrame:
    """행=날짜(yyyymmdd), 열=심볼 피벗 CSV 로드 + 정규화"""
    df = pd.read_csv(path, index_col=0)
    df.index = df.index.map(lambda x: str(x).strip())
    df.columns = df.columns.map(lambda x: str(x).strip())
    df.index = df.index.map(lambda s: re.sub(r"[-_/\.]", "", s))
    return df

def to_float(x):
    """숫자 강제 변환(문자/공백/NaN 안전 처리)"""
    try:
        return float(str(x).strip())
    except:
        return np.nan

# 1) 일별 VWAP
vwap_daily = read_pivot_csv("../data/price/vwap.csv")
if date_key not in vwap_daily.index:
    raise KeyError(f"{date_key=} not in vwap.csv index. head={list(vwap_daily.index)[:5]}")
if symbol not in vwap_daily.columns:
    raise KeyError(f"{symbol=} not in vwap.csv columns. head={list(vwap_daily.columns)[:5]}")
vwap_ref = to_float(vwap_daily.at[date_key, symbol])

# 2) 10분 VWAP 파일 모으기
price_files = sorted(
    glob.glob("../data/price_minutely/close_*.csv"),
    key=lambda p: re.findall(r"(\d{4}_\d{4})", os.path.basename(p))[0]
                  if re.findall(r"(\d{4}_\d{4})", os.path.basename(p)) else "9999_9999"
)

vwap_10m_vals = []
bad_intervals = []  # NaN/결측 발생한 파일 기록
used_intervals = []

for pfile in price_files:
    df10 = read_pivot_csv(pfile)
    if date_key not in df10.index or symbol not in df10.columns:
        continue

    val = to_float(df10.at[date_key, symbol])
    iv_match = re.findall(r"(\d{4}_\d{4})", os.path.basename(pfile))
    iv = iv_match[0] if iv_match else os.path.basename(pfile)

    if np.isnan(val) or not np.isfinite(val):
        bad_intervals.append((iv, pfile))
        continue

    vwap_10m_vals.append(val)
    used_intervals.append(iv)

if not vwap_10m_vals:
    raise RuntimeError("해당 날짜/심볼의 10분 VWAP 유효값을 찾지 못했습니다. 원본 CSV의 결측/문자값을 확인하세요.")

vwap_10m_vals = np.array(vwap_10m_vals, dtype=float)
vwap_mean = float(np.mean(vwap_10m_vals))

print(f"[{symbol}] {date_key}")
print(f"  일별 VWAP (vwap.csv): {vwap_ref:.6f}")
print(f"  10분 VWAP 평균(근사): {vwap_mean:.6f}")
print(f"  차이(%) : {(vwap_mean - vwap_ref) / vwap_ref * 100:.4f}%")
print(f"  사용된 10분 구간 수: {len(used_intervals)} / 총파일 {len(price_files)}")

if bad_intervals:
    print("\n다음 10분 구간에서 NaN/결측이 감지되었습니다. 파일 원본을 확인하세요:")
    for iv, p in bad_intervals:
        print(f"  - {iv}: {p}")

[AEGQRD] 20200103
  일별 VWAP (vwap.csv): 6177.745763
  10분 VWAP 평균(근사): 6106.439798
  차이(%) : -1.1542%
  사용된 10분 구간 수: 38 / 총파일 39

다음 10분 구간에서 NaN/결측이 감지되었습니다. 파일 원본을 확인하세요:
  - 1520_1530: ../data/price_minutely/close_1520_1530.csv
